In [23]:
import pandas as pd
import numpy as np


In [24]:
df = pd.read_excel ('First Dataset.xlsx')


In [25]:
print("--- 5 Row Sample ---")
print(df.head())


print("\n--- Dataset Info ---")
print(df.info())


print("\n--- Shape ---")
print("Rows and Columns:", df.shape)

--- 5 Row Sample ---
   customer_id first_name gender   age     city  province signup_date  \
0         1001       Reza      F  19.0    Karaj    Alborz  2025-02-19   
1         1002       Sina      M  53.0   Tehran    Tehran  2022-08-19   
2         1003      Parsa      F  31.0   Shiraz      Fars  2023-06-20   
3         1004       Sina      F  58.0  Mashhad  Khorasan  2021-11-08   
4         1005      Kimia      M  28.0  Isfahan   Isfahan  2021-10-21   

  membership_tier  purchase_count  avg_order_value  total_spending  \
0             VIP              17           121.53         2066.01   
1            Gold              12           326.47         3917.64   
2            Gold              21            59.46         1248.66   
3            Gold              23           266.15         6121.45   
4          Silver              23           169.54         3899.42   

   last_purchase_days payment_method   device discount_used  returned_items  \
0                  16           Card  An

In [26]:
numeric_cols = df.select_dtypes(include=['number']).columns
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')


text_cols = df.select_dtypes(include=['object', 'category']).columns
df[text_cols] = df[text_cols].astype(str)


date_cols = [col for col in df.columns if 'date' in col.lower() or 'time' in col.lower()]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')
    

In [27]:
df_clean = df.copy()
print("\n--- Copy created. Clean dataset shape: ---", df_clean.shape)


--- Copy created. Clean dataset shape: --- (61, 17)


In [30]:

for col in text_cols:
    df_clean[col] = df_clean[col].astype(str).str.strip()


typo_corrections = {
    'Teh': 'Tehran',
    'Tehran_': 'Tehran',
    'Isfahan': 'Esfahan'

}


for col in text_cols:
    df_clean[col] = df_clean[col].replace(typo_corrections)

print("\n--- Text cleaning completed ---")


--- Text cleaning completed ---


In [31]:
print("Duplicate rows before cleaning:", df_clean.duplicated().sum())

df_clean = df_clean.drop_duplicates()

print("Shape after removing duplicates:", df_clean.shape)



Duplicate rows before cleaning: 1
Shape after removing duplicates: (60, 17)


In [32]:
missing_data = df_clean.isnull().sum()
missing_data = missing_data[missing_data > 0]

if not missing_data.empty:
    print("Missing values per column:")
    print(missing_data)
else:
    print("No missing values found!")

Missing values per column:
age               1
total_spending    1
dtype: int64


In [33]:
median_age = df_clean['age'].median()
df_clean['age'] = df_clean['age'].fillna(median_age)

In [34]:
df_clean['total_spending'] = df_clean['total_spending'].fillna(
    df_clean['purchase_count']*df_clean['avg_order_value'])

In [35]:
print('Total missing values after cleaning:', df_clean.isnull().sum().sum())

Total missing values after cleaning: 0


In [36]:
print(" Summary Statistics ")
print(df_clean.describe().round(2))

 Summary Statistics 
       customer_id     age  purchase_count  avg_order_value  total_spending  \
count        60.00   60.00           60.00            60.00           60.00   
mean       1030.50   45.37           17.38           213.16         3715.11   
std          17.46   19.12           10.15           130.51         4273.90   
min        1001.00   19.00            0.00            27.63            0.00   
25%        1015.75   31.75           10.75           108.14         1167.04   
50%        1030.50   45.00           17.00           165.44         2128.69   
75%        1045.25   58.00           24.50           324.11         4692.15   
max        1060.00  145.00           35.00           449.81        25000.00   

       last_purchase_days  returned_items  satisfaction_score  
count               60.00           60.00               60.00  
mean               198.60            4.18                2.98  
std                 97.84            2.68                1.43  
min        

In [37]:

numeric_cols = ['age', 'purchase_count', 'avg_order_value', 'total_spending', 'last_purchase_days']

print(" Outliers count per column (IQR method) ")
for col in numeric_cols:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers_count = df_clean[(df_clean[col] < lower_bound) | (df_clean[col] > upper_bound)].shape[0]
    print(f"{col}: {outliers_count} outlier(s)")

 Outliers count per column (IQR method) 
age: 1 outlier(s)
purchase_count: 0 outlier(s)
avg_order_value: 0 outlier(s)
total_spending: 5 outlier(s)
last_purchase_days: 0 outlier(s)


In [38]:
median_age = df_clean['age'].median()
df_clean.loc[df_clean['age'] > 100, 'age'] = median_age


df_clean['total_spending'] = df_clean['purchase_count'] * df_clean['avg_order_value']

In [39]:
print("New Max Age:", df_clean['age'].max())
print("New Max Total Spending:", df_clean['total_spending'].max().round(2))

New Max Age: 65.0
New Max Total Spending: 14354.34


In [42]:

if 'signup_date' in df_clean.columns:
    df_clean['signup_date'] = pd.to_datetime(df_clean['signup_date'])
    
text_columns = df_clean.select_dtypes(include=['object']).columns

for col in text_columns:
    
    df_clean[col] = df_clean[col].astype(str).str.strip()
    

    df_clean[col] = df_clean[col].str.title()


for col in text_columns:
    print(f"\nUnique values in '{col}':")
    print(df_clean[col].unique())


Unique values in 'first_name':
['Reza' 'Sina' 'Parsa' 'Kimia' 'Amir' 'Arash' 'Neda' 'Ali' 'Mina' 'Zahra'
 'Maryam' 'Sara']

Unique values in 'gender':
['F' 'M']

Unique values in 'city':
['Karaj' 'Tehran' 'Shiraz' 'Mashhad' 'Esfahan' 'Rasht' 'Tabriz' 'Ahvaz']

Unique values in 'province':
['Alborz' 'Tehran' 'Fars' 'Khorasan' 'Esfahan' 'Gilan' 'East Azerbaijan'
 'Khuzestan']

Unique values in 'membership_tier':
['Vip' 'Gold' 'Silver' 'Bronze']

Unique values in 'payment_method':
['Card' 'Online Wallet' 'Cash']

Unique values in 'device':
['Android' 'Web' 'Iphone']

Unique values in 'discount_used':
['Yes' 'No']


In [43]:

print(df_clean.groupby(['first_name', 'gender']).size().reset_index(name='count'))

   first_name gender  count
0         Ali      F      5
1         Ali      M      1
2        Amir      F      4
3        Amir      M      2
4       Arash      F      2
5       Arash      M      2
6       Kimia      F      2
7       Kimia      M      5
8      Maryam      F      1
9      Maryam      M      3
10       Mina      F      1
11       Mina      M      2
12       Neda      F      6
13       Neda      M      2
14      Parsa      F      3
15      Parsa      M      1
16       Reza      F      5
17       Reza      M      2
18       Sara      F      1
19       Sina      F      5
20       Sina      M      3
21      Zahra      M      2


In [44]:

correct_genders = {
    'ali': 'Male',
    'amir': 'Male',
    'arash': 'Male',
    'kimia': 'Female',
    'maryam': 'Female',
    'mina': 'Female',
    'neda': 'Female',
    'parsa': 'Male',
    'reza': 'Male',
    'sara': 'Female',
    'sina': 'Male',
    'zahra': 'Female'
}


mapped_gender = df_clean['first_name'].astype(str).str.strip().str.lower().map(correct_genders)


df_clean['gender'] = mapped_gender.fillna(df_clean['gender'])


print("--- Corrected Gender Summary ---")
print(df_clean['gender'].value_counts())

--- Corrected Gender Summary ---
gender
Male      35
Female    25
Name: count, dtype: int64


In [45]:
returned_error = df_clean[df_clean['returned_items'] > df_clean['purchase_count']]

print(f"Number of rows with return error: {len(returned_error)}")
print(returned_error[['customer_id', 'first_name', 'purchase_count', 'returned_items']])

Number of rows with return error: 6
    customer_id first_name  purchase_count  returned_items
7          1008       Reza               3               8
14         1015      Kimia               3               8
23         1024       Sina               0               2
28         1029        Ali               2               4
36         1037      Zahra               1               7
55         1056      Kimia               1               4


In [46]:
df_clean.loc[df_clean['returned_items'] > df_clean['purchase_count'], 'returned_items'] = df_clean['purchase_count']

In [47]:
invalid_returns = df_clean[df_clean['returned_items'] > df_clean['purchase_count']]
print(f"Number of rows with return error after fix: {len(invalid_returns)}")

Number of rows with return error after fix: 0


In [48]:
print("==========================================")
print("         DATA HEALTH AUDIT REPORT         ")
print("==========================================")

# 1. Missing Values Check
print("1. Missing Values per Column:")
missing_series = df_clean.isnull().sum()
if missing_series.sum() == 0:
    print("   -> No missing values found! (Clean)")
else:
    print(missing_series[missing_series > 0])

# 2. Duplicate Customer IDs
duplicate_ids = df_clean[df_clean.duplicated(subset=['customer_id'], keep=False)]
print(f"\n2. Duplicate Customer IDs: {len(duplicate_ids)}")

# 3. Abnormal Age Check (e.g., age < 12 or age > 100)
if 'age' in df_clean.columns:
    abnormal_age = df_clean[(df_clean['age'] < 12) | (df_clean['age'] > 100)]
    print(f"\n3. Abnormal Age Records (<12 or >100): {len(abnormal_age)}")
    if len(abnormal_age) > 0:
        print(abnormal_age[['customer_id', 'first_name', 'age']])

# 4. Negative Financial Values
fin_cols = [c for c in ['total_spending', 'avg_order_value'] if c in df_clean.columns]
if fin_cols:
    neg_fin = df_clean[(df_clean[fin_cols] < 0).any(axis=1)]
    print(f"\n4. Negative Financial Values: {len(neg_fin)}")

# 5. Logical Check: Purchase Count vs Total Spending
if 'purchase_count' in df_clean.columns and 'total_spending' in df_clean.columns:
    zero_buy_has_spend = df_clean[(df_clean['purchase_count'] == 0) & (df_clean['total_spending'] > 0)]
    print(f"\n5. Inconsistency (0 Purchases but >0 Spending): {len(zero_buy_has_spend)}")

# 6. Logical Check: Returned Items vs Purchase Count
if 'returned_items' in df_clean.columns and 'purchase_count' in df_clean.columns:
    invalid_returns = df_clean[df_clean['returned_items'] > df_clean['purchase_count']]
    print(f"\n6. Inconsistency (Returned > Purchased): {len(invalid_returns)}")

# 7. Gender Values Standardization Check
if 'gender' in df_clean.columns:
    print(f"\n7. Unique Gender Values: {df_clean['gender'].unique().tolist()}")

print("==========================================")
print("        AUDIT COMPLETED SUCCESSFULLY      ")
print("==========================================")

         DATA HEALTH AUDIT REPORT         
1. Missing Values per Column:
   -> No missing values found! (Clean)

2. Duplicate Customer IDs: 0

3. Abnormal Age Records (<12 or >100): 0

4. Negative Financial Values: 0

5. Inconsistency (0 Purchases but >0 Spending): 0

6. Inconsistency (Returned > Purchased): 0

7. Unique Gender Values: ['Male', 'Female']
        AUDIT COMPLETED SUCCESSFULLY      


In [50]:

output_filename = "cleaned_customer_data.csv"
df_clean.to_csv(output_filename, index=False)

print(f"✅ Success! Cleaned dataset exported to '{output_filename}'")
print(f"Final Dataset Shape: {df_clean.shape[0]} rows and {df_clean.shape[1]} columns")

✅ Success! Cleaned dataset exported to 'cleaned_customer_data.csv'
Final Dataset Shape: 60 rows and 17 columns
